# SNN con convoluzioni 2x16x16

## Import Dependencies

In [1]:
!pip -q install snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 3.8 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import math
import snntorch as snn
from snntorch import surrogate
from torch.utils.data import Dataset, DataLoader
from math import gcd
from scipy import signal
from tqdm import tqdm
import os
import pandas as pd
import random
from torch.utils.data import Sampler
from collections import defaultdict

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

try:
  import google.colab
  from google.colab import runtime
  colab_env = True  # Set flag to True for Colab environment
except:
  # Set flag to False if not in Colab
  colab_env = False

try:
  from google.colab import drive
  drive.mount('/content/drive')
except:
  pass

# Path base
data_root  = Path("/content/drive/MyDrive/Federico")
load_root = data_root / "preprocessed_mapped"

# Parametri generali
fs         = 1000      # Hz
window_sec = 0.5 # with this dimension, GPU acceleration should work
win_len    = int(window_sec * fs)

n_fingers = 5
fingers = [f"finger{i}" for i in range(1, n_fingers + 1)]

sample_test = "sample1"
sample_val  = "sample1" # validation set not used
sample_train_list = ["sample2", "sample3"]

subjects = [
    "subject01_session2",
    "subject02_session2",
    "subject03_session2",
    "subject04_session2",
    "subject05_session2",
    "subject06_session2",
    "subject07_session2",
    "subject08_session2",
    "subject09_session2",
    "subject10_session2",
    "subject11_session2",
    "subject12_session2",
    "subject13_session2",
    "subject14_session2",
    "subject15_session2",
    "subject16_session2",
    "subject17_session2",
    "subject18_session2",
    "subject19_session2",
]


Device: cuda
Mounted at /content/drive


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## Funzioni per caricare le spike maps (.npz)



In [4]:
train_X, train_Y, train_fi, train_trial_id = [], [], [], []
test_X,  test_Y,  test_fi                  = [], [], []
train_win_idx, train_n_wins = [], []

for subj_idx, subj in enumerate(subjects):
    for fi, fing in enumerate(fingers):

        # -------------------- TRAIN --------------------
        for sample_train in sample_train_list:
            trial_id = subj_idx * n_fingers * len(sample_train_list) + \
                       fi * len(sample_train_list) + \
                       sample_train_list.index(sample_train)

            data = np.load(load_root / f"mapped_{subj}_{fing}_{sample_train}.npz")
            spikes_tr = data["spikes"]   # (T, 2, 16, 16)
            Y_tr      = data["force"]    # (T, 5)

            n_win  = spikes_tr.shape[0] // win_len
            T_used = n_win * win_len
            if n_win > 0:
                spikes_w = spikes_tr[:T_used].reshape(n_win, win_len, 2, 16, 16)
                Y_w      = Y_tr[:T_used].reshape(n_win, win_len, n_fingers)
                for w in range(n_win):
                    train_X.append(spikes_w[w]) # spikes
                    train_Y.append(Y_w[w])      # force
                    train_fi.append(fi)         # finger
                    train_trial_id.append(trial_id) # file path
                    train_win_idx.append(w)         # window id
                    train_n_wins.append(n_win)      # number of windows

        # -------------------- TEST (seq intera) --------------------
        data = np.load(load_root / f"mapped_{subj}_{fing}_{sample_test}.npz")
        spikes_te = data["spikes"]
        Y_te      = data["force"]

        test_X.append(spikes_te)
        test_Y.append(Y_te)
        test_fi.append(fi)

# metadati dai file (prendi dal primo)
# carica il gain della forza per il plot
meta = np.load(load_root / f"mapped_{subjects[0]}_{fingers[0]}_{sample_train_list[0]}.npz")
FORCE_GAIN  = float(meta["FORCE_GAIN"])

print("Totale finestre train:", len(train_X))
print("Totale sequenze test :", len(test_X))
print(f"FORCE_GAIN={FORCE_GAIN}")

Totale finestre train: 9500
Totale sequenze test : 95
FORCE_GAIN=100.0



## Dataset PyTorch (windowed per train/val, full sequence per test)

In [5]:
class SpikeForceWindowDataset(Dataset):
    def __init__(self, Xlist, Ylist, filist, trial_idlist, win_idxlist, n_wins_list):
        self.Xlist         = np.asarray(Xlist, dtype=np.float32)
        self.Ylist         = np.asarray(Ylist, dtype=np.float32)
        self.filist        = list(filist)
        self.trial_idlist  = list(trial_idlist)
        self.win_idxlist   = list(win_idxlist)   # posizione dentro il trial (0..49)
        self.n_wins_list   = list(n_wins_list)   # quante finestre ha quel trial

    def __len__(self):
        return len(self.Xlist)

    def __getitem__(self, idx):
        x        = torch.from_numpy(self.Xlist[idx])
        y        = torch.from_numpy(self.Ylist[idx])
        fi       = int(self.filist[idx])
        trial_id = int(self.trial_idlist[idx])
        win_idx  = int(self.win_idxlist[idx])
        n_wins   = int(self.n_wins_list[idx])
        return x, y, fi, trial_id, win_idx, n_wins

class SpikeForceSeqDataset(Dataset):
    def __init__(self, X_list, Y_list, fi_list):
        assert len(X_list) == len(Y_list) == len(fi_list)
        self.X_list  = [np.asarray(x, dtype=np.float32) for x in X_list]
        self.Y_list  = [np.asarray(y, dtype=np.float32) for y in Y_list]
        self.fi_list = list(fi_list)

    def __len__(self):
        return len(self.X_list)

    def __getitem__(self, idx):
        x  = torch.from_numpy(self.X_list[idx])   # (T,2,16,16)
        y  = torch.from_numpy(self.Y_list[idx])   # (T,5)
        fi = int(self.fi_list[idx])               # 0..4
        return x, y, fi


# ---- istanzia dataset ----
train_dataset = SpikeForceWindowDataset(train_X, train_Y, train_fi, train_trial_id,train_win_idx, train_n_wins)
test_dataset = SpikeForceSeqDataset(test_X, test_Y, test_fi)

print("len(train_dataset):", len(train_dataset))
print("len(test_dataset) :", len(test_dataset))

len(train_dataset): 9500
len(test_dataset) : 95


In [6]:
class TemporalParallelSampler(Sampler):
    def __init__(self, dataset, batch_size, shuffle_trials=True):
        self.batch_size     = batch_size
        self.shuffle_trials = shuffle_trials

        win2indices = defaultdict(list)
        for i, w in enumerate(dataset.win_idxlist):
            win2indices[w].append(i)

        self.win2indices = dict(win2indices)
        self.win_keys    = sorted(self.win2indices.keys())

    def __iter__(self):
        for w in self.win_keys:
            indices = self.win2indices[w].copy()
            if self.shuffle_trials:
                random.shuffle(indices)
            for start in range(0, len(indices) - self.batch_size + 1, self.batch_size):
                yield indices[start : start + self.batch_size]

    def __len__(self):
        total = 0
        for indices in self.win2indices.values():
            total += len(indices) // self.batch_size
        return total


## Dataloader

In [7]:
batch_size = 190

train_sampler = TemporalParallelSampler(train_dataset, batch_size=batch_size, shuffle_trials=False)
train_loader = DataLoader(train_dataset,batch_sampler=train_sampler)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches :", len(test_loader))

Train batches: 50
Test batches : 95


## RETE

In [ ]:
class ConvSNN_TC9(nn.Module):
    """
    TC9:
    Conv3x3 valid -> LIF          (16 -> 14)
    Conv3x3 same  -> LIF          (14 -> 14)
    MaxPool2x2 s=2                (14 -> 7)
    Conv3x3 valid -> LIF          (7 -> 5)
    Flatten (16*5*5 = 400)
    Dense 400->5 -> integratore
    Dimensioni: 16 -> 14 -> 14 -> 7 -> 5
    """
    def __init__(
        self,
        in_channels=2,
        beta_val=0.9,
        threshold_hidden=0.3,
        threshold_out=1e9,
        ch1=16,
        ch2=8,
        ch3=16,
        out_dim=5,
        input_h=16,
        input_w=16,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.input_h = input_h
        self.input_w = input_w

        spike_grad = surrogate.fast_sigmoid()

        # L1: 16 -> 14
        self.conv1 = nn.Conv2d(in_channels, ch1,
                               kernel_size=3, stride=1, padding=0, bias=False)
        self.H1 = input_h      # 14
        self.W1 = input_w      # 14
        self.lif1 = snn.Leaky(
            beta=beta_val, threshold=threshold_hidden,
            learn_beta=True, learn_threshold=False,
            spike_grad=spike_grad, reset_mechanism="subtract"
        )

        # L2: 14 -> 14 (same: padding=1, stride=1)
        self.conv2 = nn.Conv2d(ch1, ch2,
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.H2 = self.H1 - 2          # 14
        self.W2 = self.W1 - 2          # 14
        self.lif2 = snn.Leaky(
            beta=beta_val, threshold=threshold_hidden,
            learn_beta=True, learn_threshold=False,
            spike_grad=spike_grad, reset_mechanism="subtract"
        )

        # MaxPool: 14 -> 7 (kernel=2, stride=2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.H2p = self.H2 // 2    # 7
        self.W2p = self.W2 // 2    # 7

        # L3: 7 -> 5
        self.conv3 = nn.Conv2d(ch2, ch3,
                               kernel_size=3, stride=1, padding=0, bias=False)
        self.H3 = self.H2p - 2     # 5
        self.W3 = self.W2p - 2     # 5
        self.lif3 = snn.Leaky(
            beta=beta_val, threshold=threshold_hidden,
            learn_beta=True, learn_threshold=False,
            spike_grad=spike_grad, reset_mechanism="subtract"
        )

        flat_dim = ch3 * self.H3 * self.W3  # 16*5*5 = 400

        # Dense OUT diretto: 400 -> 5
        self.fc_out = nn.Linear(flat_dim, out_dim, bias=False)
        beta_out = torch.ones(out_dim) * 0.99
        self.li_out = snn.Leaky(
            beta=beta_out, threshold=threshold_out,
            learn_beta=True, learn_threshold=False,
            spike_grad=spike_grad, reset_mechanism="none"
        )

    def forward(self, x, mem_init=None):
        B, T, C, H, W = x.shape
        x = x.to(next(self.parameters()).device)

        # to initizilize neurons internal state to zero or to another value
        if mem_init is not None:
            mem1    = mem_init["mem1"].to(x.device).clone()
            mem2    = mem_init["mem2"].to(x.device).clone()
            mem3    = mem_init["mem3"].to(x.device).clone()
            mem_out = mem_init["mem_out"].to(x.device).clone()
        else:
            mem1    = torch.zeros((B, self.conv1.out_channels, self.H1, self.W1), device=x.device)
            mem2    = torch.zeros((B, self.conv2.out_channels, self.H2, self.W2), device=x.device)
            mem3    = torch.zeros((B, self.conv3.out_channels, self.H3, self.W3), device=x.device)
            mem_out = torch.zeros((B, self.fc_out.out_features), device=x.device)

        mem_rec = []
        for step in range(T):
            x_t = x[:, step]
            cur1 = self.conv1(x_t)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.conv2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2p = self.pool2(spk2)
            cur3 = self.conv3(spk2p)
            spk3, mem3 = self.lif3(cur3, mem3)
            feat = spk3.view(B, -1)
            cur_out = self.fc_out(feat)
            _, mem_out = self.li_out(cur_out, mem_out)
            mem_rec.append(mem_out)

        # restituisce anche gli stati finali per il carry
        mem_final = {
            "mem1":    mem1.detach().clone(),
            "mem2":    mem2.detach().clone(),
            "mem3":    mem3.detach().clone(),
            "mem_out": mem_out.detach().clone(),
        }

        return torch.stack(mem_rec, dim=0), mem_final

# istanzia il modello
model = ConvSNN_TC9(in_channels=2).to(device)
print(model)

ConvSNN_TC8(
  (conv1): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (lif1): Leaky()
  (conv2): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (lif2): Leaky()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), bias=False)
  (lif3): Leaky()
  (fc_out): Linear(in_features=576, out_features=5, bias=False)
  (li_out): Leaky()
)


## Training Loop

In [ ]:
# Cartella dove salvare i modelli
model_dir = data_root / "models"
model_dir.mkdir(parents=True, exist_ok=True)

best_ckpt_path = model_dir / "TC15_2_3.pt"
last_ckpt_path = model_dir / "TC15_2_3_last.pt"

In [ ]:
initial_lr       = 3e-4
lr_factor        = 1.0/3
patience_epochs  = 30
max_lr_updates   = 3
max_epochs       = 800

optimizer = torch.optim.Adam(model.parameters(), lr=initial_lr)

best_train_loss = float("inf")
train_loss_hist = []
val_loss_hist   = []

epochs_since_improve = 0
lr_updates = 0
learning_rate = initial_lr
epoch_last = None

# errors on the active figers weights 5x errors on each non-active finger
def weighted_mse_loss(out_t_b_f, y_t_b_f, fi_b, n_fingers=5, w_active=5.0):
    B = y_t_b_f.shape[1]
    w = torch.ones((B, n_fingers), device=y_t_b_f.device, dtype=y_t_b_f.dtype)
    w[torch.arange(B, device=y_t_b_f.device), fi_b] = w_active
    w = w.view(1, B, n_fingers)
    diff = out_t_b_f - y_t_b_f
    return (w * diff * diff).mean()

# ------------------------- RESUME -------------------------

if os.path.exists(last_ckpt_path):
    print("🔁 Resume from:", last_ckpt_path)
    ckpt = torch.load(last_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    best_train_loss      = ckpt["best_train_loss"]
    epochs_since_improve = ckpt["epochs_since_improve"]
    lr_updates           = ckpt["lr_updates"]
    train_loss_hist      = ckpt["train_loss_hist"]
    val_loss_hist        = ckpt["val_loss_hist"]
    epoch_last           = ckpt["epoch_last"]
    learning_rate        = ckpt["learning_rate"]

print(f"Start | Current LR: {optimizer.param_groups[0]['lr']:.2e}")

model.train()
# ------------------------- TRAIN LOOP -------------------------

while True:

    if epoch_last is not None and (epoch_last + 1) >= max_epochs:
        break

    if epoch_last is None:
        epoch_last = 0
    else:
        epoch_last += 1

    print(f"\nEpoch {epoch_last}/{max_epochs-1}")

    # =========================
    # TRAIN
    # =========================

    loss_values = []

    mem_carry = defaultdict(lambda: None)

    train_bar = tqdm(train_loader, desc="Train", leave=False)

    for X_batch, Y_batch, fi_batch, trial_id_batch, win_idx_batch, n_wins_batch in train_bar:

        X_batch        = X_batch.to(device)
        Y_batch        = Y_batch.to(device)
        fi_batch       = fi_batch.to(device).long()
        trial_id_batch = trial_id_batch.to(device).long()

        B = X_batch.shape[0]

        mem_init = {
            "mem1":    torch.zeros(B, model.conv1.out_channels, model.H1, model.W1, device=device),
            "mem2":    torch.zeros(B, model.conv2.out_channels, model.H2, model.W2, device=device),
            "mem3":    torch.zeros(B, model.conv3.out_channels, model.H3, model.W3, device=device),
            "mem_out": torch.zeros(B, model.fc_out.out_features, device=device),
        }

        for i in range(B):
            tid     = trial_id_batch[i].item()
            win_i   = win_idx_batch[i].item()

            if win_i == 0:
                # reset carry
                mem_carry[tid] = None

            if win_i > 0 and mem_carry[tid] is not None:
                mem_init["mem1"][i]    = mem_carry[tid]["mem1"]
                mem_init["mem2"][i]    = mem_carry[tid]["mem2"]
                mem_init["mem3"][i]    = mem_carry[tid]["mem3"]
                mem_init["mem_out"][i] = mem_carry[tid]["mem_out"]

        output, mem_final = model(X_batch, mem_init=mem_init)

        for i in range(B):
            tid = trial_id_batch[i].item()
            mem_carry[tid] = {
                "mem1":    mem_final["mem1"][i].detach(),
                "mem2":    mem_final["mem2"][i].detach(),
                "mem3":    mem_final["mem3"][i].detach(),
                "mem_out": mem_final["mem_out"][i].detach(),
            }

        Y_t_b_f = Y_batch.permute(1, 0, 2)
        loss = weighted_mse_loss(output, Y_t_b_f, fi_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_values.append(loss.item())
        train_bar.set_postfix(loss=f"{loss.item():.3e}")

    train_loss = sum(loss_values) / len(loss_values)
    train_loss_hist.append(train_loss)

    # =========================
    # SAVE LAST
    # =========================

    last_ckpt = {
        "epoch_last":           epoch_last,
        "model_state":          model.state_dict(),
        "optimizer_state":      optimizer.state_dict(),
        "best_train_loss":      best_train_loss,
        "epochs_since_improve": epochs_since_improve,
        "lr_updates":           lr_updates,
        "train_loss_hist":      train_loss_hist,
        "val_loss_hist":        val_loss_hist,
        "learning_rate":        learning_rate,
    }

    torch.save(last_ckpt, last_ckpt_path)

    # =========================
    # BEST ON TRAIN
    # =========================

    if train_loss < best_train_loss:
        best_train_loss = train_loss
        epochs_since_improve = 0
        torch.save(last_ckpt, best_ckpt_path)
    else:
        epochs_since_improve += 1

    # =========================
    # LR DECAY + EARLY STOP
    # =========================

    if epochs_since_improve >= patience_epochs:

        if lr_updates < max_lr_updates:
            old_lr = learning_rate
            learning_rate *= lr_factor
            for g in optimizer.param_groups:
                g["lr"] = learning_rate
            lr_updates += 1
            epochs_since_improve = 0
            print(f"Reducing LR: {old_lr:.2e} → {learning_rate:.2e}")

        else:
            print("⛔ Early stopping triggered.")
            break

    print(
        f"Train: {train_loss:.4e} | "
        f"BestTrain: {best_train_loss:.4e} | "
        f"LR: {learning_rate:.2e} | "
        f"Patience: {epochs_since_improve}/{patience_epochs}"
    )

# ------------------------- LOAD BEST -------------------------

if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])

print("✅ Training finished.")

# ------------------------- PLOT -------------------------

plt.figure(figsize=(8, 4))
plt.semilogy(train_loss_hist, label="Train")
if val_loss_hist:
    plt.semilogy(val_loss_hist, label="Val")
plt.xlabel("Epoch")
plt.ylabel("Weighted MSE (log scale)")
plt.legend()
plt.grid(True)
plt.show()

Start | Current LR: 3.00e-04

Epoch 0/799


Train: 8.6138e+01 | BestTrain: 8.6138e+01 | LR: 3.00e-04 | Patience: 0/30

Epoch 1/799


Train: 6.7064e+01 | BestTrain: 6.7064e+01 | LR: 3.00e-04 | Patience: 0/30

Epoch 2/799


Train: 6.5074e+01 | BestTrain: 6.5074e+01 | LR: 3.00e-04 | Patience: 0/30

Epoch 3/799


Train: 6.1782e+01 | BestTrain: 6.1782e+01 | LR: 3.00e-04 | Patience: 0/30

Epoch 4/799


...

Train: 1.3699e+01 | BestTrain: 1.0732e+01 | LR: 3.00e-04 | Patience: 11/30

Epoch 218/799


Train: 1.3126e+01 | BestTrain: 1.0732e+01 | LR: 3.00e-04 | Patience: 12/30

Epoch 219/799


Train: 1.1899e+01 | BestTrain: 1.0732e+01 | LR: 3.00e-04 | Patience: 13/30

Epoch 220/799


Train: 1.1757e+01 | BestTrain: 1.0732e+01 | LR: 3.00e-04 | Patience: 14/30

Epoch 221/799


Train:  74%|███████▍  | 37/50 [01:16<00:27,  2.12s/it, loss=1.460e+01]

## TEST VISIVO

In [ ]:
# ---------------------------
# Carica checkpoint
# ---------------------------
ckpt = torch.load(best_ckpt_path, map_location=device)

if isinstance(ckpt, dict) and "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"])
else:
    model.load_state_dict(ckpt)

model.eval()
print("✅ Loaded:", best_ckpt_path)

# Seleziona soggetto + trial-finger
subj_test  = subjects[0]   # es: "subject01_session2"
finger_idx = 0             # 0..4 (0=finger1,...,4=finger5)

seq_index = subjects.index(subj_test) * n_fingers + finger_idx
print("Using test_dataset index:", seq_index, "| subj:", subj_test, "| trial finger:", finger_idx+1)

X_seq_np, Y_seq_np, *_ = test_dataset[seq_index]

X_seq = X_seq_np.unsqueeze(0).to(device)       # (1,T,8,8,8)
Y_seq = Y_seq_np.unsqueeze(0).to(device)       # (1,T,5)

# Intervallo temporale: 0 → 25 s
t_start_sec = 0.0
t_end_sec   = 25.0

start = int(t_start_sec * fs)
end   = min(int(t_end_sec * fs), X_seq.shape[1])

X_seq_sel = X_seq[:, start:end]
Y_seq_sel = Y_seq[:, start:end]

t_axis = np.arange(end - start) / fs

# Forward
with torch.no_grad():
    out_t_b_f, _ = model(X_seq_sel)             # (T,1,5) — ignora mem_final
    pred_z = out_t_b_f[:, 0].cpu().numpy()      # (T,5)
    true_z = Y_seq_sel[0].cpu().numpy()         # (T,5)

# De-standardizzazione (TUTTE le dita)
pred_real = (pred_z / FORCE_GAIN)
true_real = (true_z / FORCE_GAIN)


# Limiti globali asse Y (TUTTE le dita)
y_min = min(true_real.min(), pred_real.min())
y_max = max(true_real.max(), pred_real.max())

# margine (5%)
margin = 0.05 * (y_max - y_min)
y_min -= margin
y_max += margin

# Plot: tutte le dita (0–25 s) con stesso asse Y
for i in range(n_fingers):
    plt.figure(figsize=(12,4))
    plt.plot(t_axis, true_real[:, i], label=f"True finger{i+1}", alpha=0.9)
    plt.plot(t_axis, pred_real[:, i], label=f"Pred finger{i+1}", alpha=0.7)

    plt.xlabel("Tempo [s]")
    plt.ylabel("Forza")
    plt.ylim(y_min, y_max)   # 🔴 asse Y condiviso

    plt.title(
        f"TEST 0–25 s | subj={subj_test} | "
        f"trial finger={finger_idx+1} | plotted finger={i+1}"
    )
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def pearson_corr(y, yhat, eps=1e-8):
    y = y.float(); yhat = yhat.float()
    ym = y.mean(); yhm = yhat.mean()
    num = ((y-ym)*(yhat-yhm)).sum()
    den = torch.sqrt(((y-ym)**2).sum() * ((yhat-yhm)**2).sum() + eps)
    return num / den

def r2_score(y, yhat, eps=1e-8):
    y = y.float(); yhat = yhat.float()
    ss_res = ((y - yhat)**2).sum()
    ss_tot = ((y - y.mean())**2).sum()
    return 1.0 - ss_res / (ss_tot + eps)

# --- load trained model ---
ckpt = torch.load(best_ckpt_path, map_location=device)
if isinstance(ckpt, dict) and "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"])
else:
    model.load_state_dict(ckpt)

model.eval()
cors, r2s = [], []

with torch.no_grad():
    for X_seq, Y_seq, fi in test_loader:
        X_seq = X_seq.to(device)   # (1,T,2,16,16)
        Y_seq = Y_seq.to(device)   # (1,T,5)
        fi = fi.to(device).long()  # (1,)

        k = fi.item()                 # <-- dito attivo

        out_t_b_f, _ = model(X_seq)
        pred = out_t_b_f[:, 0, k]     # (T,)
        true = Y_seq[0, :, k]         # (T,)

        cors.append(pearson_corr(true, pred).item())
        r2s.append(r2_score(true, pred).item())

print("FINAL TEST (moving finger) COR mean:", sum(cors)/max(1,len(cors)))
print("FINAL TEST (moving finger) R2  mean:", sum(r2s)/max(1,len(r2s)))